# 01 - The Attention Mechanism

**Goal:** Understand attention - the core idea behind transformers.

---

## Why Attention?

**CNNs** look at local regions (3x3 kernels). To see the whole image, you need many layers.

**Attention** lets each part of the input directly look at every other part in one step.

```
CNN:       Each pixel sees its neighbors → stack layers → eventually sees whole image
Attention: Each pixel can directly attend to ANY other pixel in one layer
```

## The Intuition

Imagine reading a sentence:

> "The cat sat on the mat because **it** was tired."

What does "it" refer to? You need to look back at "cat".

**Attention** = learning which parts to focus on.

For images: when segmenting a vertebra, it helps to know where other vertebrae are.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

## Query, Key, Value

Attention uses three projections of the input:

| Name | Analogy | Purpose |
|------|---------|--------|
| **Query (Q)** | "What am I looking for?" | The current position asking for info |
| **Key (K)** | "What do I have?" | What each position offers |
| **Value (V)** | "What's my content?" | The actual information to retrieve |

**Attention formula:**
```
Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d)) @ V
```

1. Compute similarity: `Q @ K.T` (dot product between queries and keys)
2. Scale: divide by `sqrt(d)` to keep gradients stable
3. Normalize: `softmax` to get attention weights (sum to 1)
4. Aggregate: weighted sum of values

In [ ]:
def simple_attention(query, key, value):
    """
    Simple scaled dot-product attention.
    
    query: (seq_len, d_k)
    key:   (seq_len, d_k)
    value: (seq_len, d_v)
    """
    d_k = query.shape[-1]
    
    # Step 1: Compute attention scores
    scores = query @ key.T  # (seq_len, seq_len)
    
    # Step 2: Scale
    scores = scores / np.sqrt(d_k)
    
    # Step 3: Softmax to get attention weights
    weights = F.softmax(torch.tensor(scores), dim=-1).numpy()
    
    # Step 4: Weighted sum of values
    output = weights @ value
    
    return output, weights

# Example: 4 positions, dimension 8
seq_len, d_k = 4, 8

# Random Q, K, V (in practice, these come from linear projections)
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = simple_attention(Q, K, V)

print(f"Query shape: {Q.shape}")
print(f"Key shape: {K.shape}")
print(f"Value shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"\nAttention weights (each row sums to 1):\n{weights.round(3)}")

In [ ]:
# Visualize attention weights
plt.figure(figsize=(6, 5))
plt.imshow(weights, cmap='Blues')
plt.colorbar(label='Attention weight')
plt.xlabel('Key position (attending to)')
plt.ylabel('Query position (attending from)')
plt.title('Attention Weights\n(each row shows where that position looks)')
plt.xticks(range(4), ['Pos 0', 'Pos 1', 'Pos 2', 'Pos 3'])
plt.yticks(range(4), ['Pos 0', 'Pos 1', 'Pos 2', 'Pos 3'])
plt.tight_layout()
plt.show()

## Self-Attention

**Self-attention** = Q, K, V all come from the same input.

Each position attends to all other positions in the same sequence.

In [ ]:
class SelfAttention(nn.Module):
    """Single-head self-attention."""
    
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Linear projections for Q, K, V
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        
        Q = self.query(x)  # (batch, seq_len, embed_dim)
        K = self.key(x)
        V = self.value(x)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq_len, seq_len)
        scores = scores / (self.embed_dim ** 0.5)
        
        # Attention weights
        weights = F.softmax(scores, dim=-1)
        
        # Weighted sum of values
        output = torch.matmul(weights, V)
        
        return output, weights

# Test
attn = SelfAttention(embed_dim=64)
x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, embed_dim=64

output, weights = attn(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Attention weights: {weights.shape}")

## Multi-Head Attention

Instead of one attention, use multiple "heads" that learn different patterns:

- Head 1 might learn to look at nearby positions
- Head 2 might learn to look at similar features
- Head 3 might learn structural relationships

Then concatenate all heads.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention."""
    
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        batch, seq_len, _ = x.shape
        
        # Project and reshape for multi-head
        Q = self.query(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # Now: (batch, num_heads, seq_len, head_dim)
        
        # Attention per head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(weights, V)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch, seq_len, self.embed_dim)
        
        # Final projection
        output = self.out(attn_output)
        
        return output

# Test
mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x = torch.randn(2, 10, 64)

output = mha(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in mha.parameters()):,}")

## PyTorch's Built-in

PyTorch provides `nn.MultiheadAttention`:

In [ ]:
# PyTorch built-in (note: expects seq_len first by default)
mha_pytorch = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)

x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, embed_dim=64
output, attn_weights = mha_pytorch(x, x, x)  # Q, K, V all same for self-attention

print(f"Output: {output.shape}")
print(f"Attention weights: {attn_weights.shape}")

## Attention for Images

To apply attention to images, we need to convert the 2D image to a sequence:

```
Image: (H, W)  →  Flatten  →  Sequence: (H*W,)
```

Then each pixel can attend to every other pixel.

**Problem:** For a 224x224 image, that's 50,176 positions. Attention is O(n²), so this is expensive!

In [ ]:
# Attention complexity
def attention_complexity(seq_len):
    return seq_len ** 2

sizes = {
    '32x32': 32 * 32,
    '64x64': 64 * 64,
    '224x224': 224 * 224,
    '96x96x96 (3D)': 96 * 96 * 96,
}

print("Attention complexity (number of pairs):")
for name, seq_len in sizes.items():
    complexity = attention_complexity(seq_len)
    print(f"  {name}: {seq_len:,} positions → {complexity:,} pairs")

## Solutions to the Complexity Problem

| Approach | How it works | Example |
|----------|-------------|--------|
| **Patches** | Divide image into patches, attend between patches | ViT (16x16 patches) |
| **Windows** | Attend only within local windows | Swin Transformer |
| **Axial** | Attend along each axis separately | Axial Attention |

**Swin Transformer** (what your model uses) uses **windowed attention** - we'll cover this in detail.

## Positional Encoding

Attention doesn't know position - it treats input as a set, not a sequence.

**Positional encoding** adds position information:

```
input = token_embedding + position_embedding
```

In [ ]:
# Simple learnable positional embedding
class PositionalEmbedding(nn.Module):
    def __init__(self, seq_len, embed_dim):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, embed_dim) * 0.02)
    
    def forward(self, x):
        return x + self.pos_embed

# Test
pos_embed = PositionalEmbedding(seq_len=100, embed_dim=64)
x = torch.randn(2, 100, 64)
print(f"Input: {x.shape}")
print(f"With position: {pos_embed(x).shape}")

## Summary

| Concept | What it means |
|---------|---------------|
| **Attention** | Weighted aggregation based on similarity |
| **Query, Key, Value** | Different projections of input for computing attention |
| **Self-attention** | Q, K, V all from same input |
| **Multi-head** | Multiple parallel attention heads |
| **Positional encoding** | Adding position information |
| **Complexity** | O(n²) - problem for large images |

**Key insight:** Attention lets each position directly access any other position, unlike CNNs which need many layers for large receptive fields.

**Next:** Vision Transformer (ViT) - applying attention to images.